# Stage 1 — baseline evaluation and failure analysis

Loads the logged runs under `results/runs/`, prints the summary tables, draws the figures, and shows how to drill into individual failures. Run from the repo root (`jupyter notebook notebooks/01_baseline_eval.ipynb`).

In [ ]:
import os, sys, json
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import pandas as pd
from pathlib import Path
from slm_agent_lab.eval.report import load_runs, summarize, write_report
runs = sorted(p for p in Path("results/runs").glob("*") if (p / "trajectories.jsonl").exists())
df = load_runs(runs)
summary = summarize(df)
pd.DataFrame({m: {k: v for k, v in s.items() if not isinstance(v, dict)} for m, s in summary["models"].items()}).T

In [ ]:
# accuracy by category and failure taxonomy as tables
display(pd.DataFrame({m: s["accuracy_by_category"] for m, s in summary["models"].items()}).T.round(2))
display(pd.DataFrame({m: s["failure_labels"] for m, s in summary["models"].items()}).T.fillna(0).astype(int))

In [ ]:
write_report(runs, "results/report", "docs/figures")
from IPython.display import Image, display
for f in ["accuracy_by_category.png", "failure_taxonomy.png", "accuracy_vs_throughput.png"]:
    display(Image(f"docs/figures/{f}"))

## Drill into failures
Every trajectory is logged with the raw model output, the parsed call, the tool result and the taxonomy label, so a failure category can be opened and read.

In [ ]:
!python scripts/inspect_run.py {runs[0]} --summary
!python scripts/inspect_run.py {runs[0]} --label wrong_answer_after_correct_tools --n 2

In [ ]:
# How do models talk to tools? (dialect of the emitted tool calls) and does OOD hurt?
display(df.groupby("model")["call_sources"].value_counts().unstack(fill_value=0))
display(df.groupby(["model", "ood"])["correct"].mean().unstack().round(3))